# MLP improvement on unseen airfoils

Use a fresh Colab CPU runtime and run all cells. This notebook makes three independent fits: a fresh 100-iteration control, then 200- and 400-iteration budgets, with architecture, scaling, batch size, learning rate, and seed held fixed. The control is rerun so diagnostics and the runtime are comparable. Budgets are upper limits: the optimizer may stop earlier if its training-loss stopping criterion is met.

**The existing split already holds out entire airfoil IDs.** All operating conditions for a geometry ID stay in one partition. Validation guides model selection, so it is not an untouched final test. Duplicate/near-duplicate shapes under different IDs have not been independently audited; airfoils outside the NASA dataset or operating envelope still require separate validation.

The default run uses validation only and leaves the test set untouched. More training is an experiment, not a guarantee of R-squared 0.99. Models remain in Drive; only small reports and plots are imported locally.

In [ ]:
from pathlib import Path
import subprocess
import sys
from google.colab import drive

PROJECT_ROOT = Path('/content/airfoil-ml-surrogate')
if not PROJECT_ROOT.exists():
    subprocess.run(['git', 'clone', 'https://github.com/gaurav-zz/ML-for-CFD.git', str(PROJECT_ROOT)], check=True)
if not (PROJECT_ROOT / '.git').exists():
    raise FileNotFoundError('Expected a Git checkout. Use a fresh Colab runtime.')
if subprocess.check_output(['git', 'status', '--porcelain'], cwd=PROJECT_ROOT, text=True).strip():
    raise RuntimeError('The checkout has local edits. Preserve them and use a fresh runtime.')
previous_revision = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=PROJECT_ROOT, text=True).strip()
subprocess.run(['git', 'pull', '--ff-only', 'origin', 'main'], cwd=PROJECT_ROOT, check=True)
SOURCE_REVISION = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=PROJECT_ROOT, text=True).strip()
if SOURCE_REVISION != previous_revision and any(name == 'src' or name.startswith('src.') for name in sys.modules):
    raise RuntimeError('Source changed after import. Restart the session and run all cells again.')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PROJECT_ROOT / 'requirements.txt')], check=True)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f'Source revision: {SOURCE_REVISION}')

## Configuration

`BASELINE_RUN = None` chooses the latest baseline run.json. The CSV hash, feature order, and exact saved geometry partitions are verified before fitting. Targets remain those of the baseline (currently Cl and Cd); adding Cm should be a separate controlled experiment after checking its labels.

For a completed improvement experiment, set `EXISTING_RUN` to its Drive directory. The notebook restores that experiment's baseline and skips all fitting, allowing reports or final testing without retraining. `RUN_FINAL_TEST` must stay false during tuning.

In [ ]:
drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/airfoil-ml-surrogate')
PROCESSED_PATH = DRIVE_ROOT / 'data' / 'processed' / 'airfoil_learning.csv'
BASELINE_RUN = None
EXISTING_RUN = None
RUN_FINAL_TEST = False

In [ ]:
from copy import deepcopy
from datetime import datetime, timezone
from importlib.metadata import version
import gc
import json
import joblib
import numpy as np
import pandas as pd
from src.training.comparison import (
    MLP_IMPROVEMENT_CONFIGS, compare_models, evaluate_selected_models,
    load_baseline_split, model_artifact_path,
)

previous_report = None
if EXISTING_RUN is not None:
    EXISTING_RUN = Path(EXISTING_RUN)
    previous_report = json.loads((EXISTING_RUN / 'run.json').read_text())
    if previous_report['run_metadata'].get('experiment_type') != 'mlp_improvement':
        raise ValueError('EXISTING_RUN must be an MLP improvement experiment.')
    if previous_report['status'] not in ('completed', 'completed_with_failures'):
        raise ValueError('EXISTING_RUN must be completed; interrupted fits are not automatically resumed.')
    if BASELINE_RUN is None:
        BASELINE_RUN = previous_report['run_metadata']['baseline_run']
if BASELINE_RUN is None:
    candidates = sorted((DRIVE_ROOT / 'experiments' / 'baseline').glob('*/run.json'))
    if not candidates:
        raise FileNotFoundError('Complete notebook 02 first; no baseline run.json was found.')
    BASELINE_RUN = candidates[-1]
BASELINE_RUN = Path(BASELINE_RUN)
print(f'Restoring frozen baseline: {BASELINE_RUN}', flush=True)
split, feature_columns, target_columns, baseline_metadata = load_baseline_split(PROCESSED_PATH, BASELINE_RUN)
if previous_report is not None:
    for key in ('dataset', 'feature_columns', 'split_geometries', 'split_rows'):
        if previous_report['run_metadata'][key] != baseline_metadata[key]:
            raise ValueError(f'Existing experiment does not match the baseline: {key}')
    if previous_report['target_names'] != target_columns:
        raise ValueError('Existing experiment target names/order do not match.')
    if previous_report['run_metadata']['packages']['scikit-learn'] != version('scikit-learn'):
        raise RuntimeError('Use the saved scikit-learn version before loading the existing models, then restart the session.')
X_train, y_train = split.train[feature_columns], split.train[target_columns]
X_validation, y_validation = split.validation[feature_columns], split.validation[target_columns]
for partition in ('train', 'validation', 'test'):
    frame = getattr(split, partition)
    print(f'{partition}: {len(frame):,} rows, {frame.geometry_id.nunique():,} geometries')
print('No geometry ID crosses partitions. Test data are reserved, not used for model selection.')

In [ ]:
MODEL_CONFIGS = deepcopy(previous_report['model_configs'] if previous_report else MLP_IMPROVEMENT_CONFIGS)
display(pd.DataFrame(MODEL_CONFIGS))

## Run the controlled experiment

Each variant has its own ID, model file, warnings, loss curve, and validation scores. Models run sequentially. Incomplete or failed fits do not overwrite other variants; ordinary failures are logged and later variants continue. Keep all settings except the iteration budget fixed for this first experiment. A higher budget may take longer and is not necessarily better.

`early_stopping=False` disables the internal random validation holdout, not the training-loss stopping criterion. Increasing `max_iter` alone does not override tolerance or patience. Compare actual iterations, not just the variant IDs; runs can stop at the same iteration and produce identical scores. Older saved runs may not contain the newly recorded tolerance/patience fields.

In [ ]:
if EXISTING_RUN is not None:
    RUN_OUTPUT = EXISTING_RUN
    validation_results = pd.read_csv(RUN_OUTPUT / 'comparison.csv')
    print(f'Using saved experiment without retraining: {RUN_OUTPUT}')
else:
    run_started = datetime.now(timezone.utc)
    RUN_OUTPUT = DRIVE_ROOT / 'experiments' / 'mlp_improvement' / run_started.strftime('%Y%m%dT%H%M%S%fZ')
    run_metadata = {
        'experiment_type': 'mlp_improvement', 'created_at_utc': run_started.isoformat(),
        'git_commit': SOURCE_REVISION, 'python_version': sys.version,
        'packages': {name: version(name) for name in ['numpy', 'pandas', 'scikit-learn', 'joblib', 'matplotlib']},
        'baseline_run': str(BASELINE_RUN),
        'dataset': baseline_metadata['dataset'], 'feature_columns': feature_columns,
        'split_geometries': baseline_metadata['split_geometries'], 'split_rows': baseline_metadata['split_rows'],
    }
    validation_results = compare_models(
        X_train, y_train, X_validation, y_validation, target_columns, RUN_OUTPUT,
        model_configs=MODEL_CONFIGS, run_metadata=run_metadata,
        train_groups=split.train['geometry_id'], validation_groups=split.validation['geometry_id'],
    )
print(f'Artifacts: {RUN_OUTPUT}')

In [ ]:
report = json.loads((RUN_OUTPUT / 'run.json').read_text())
if validation_results.empty:
    raise RuntimeError(f'No variants succeeded. Inspect {RUN_OUTPUT / "run.json"}.')
display(validation_results.sort_values(['target', 'rmse', 'model']).set_index(['target', 'model']))
display(pd.DataFrame([
    {'model': name, 'status': result['status'],
     'iterations': result.get('training', {}).get('n_iter'),
     'budget': result.get('training', {}).get('max_iter'),
     'hit_limit': result.get('training', {}).get('hit_iteration_limit'),
     'tolerance': result.get('training', {}).get('tol', 'not recorded'),
     'patience': result.get('training', {}).get('n_iter_no_change', 'not recorded'),
     'warnings': '; '.join(item['message'] for item in result['warnings'])}
    for name, result in report['results'].items()
]))
by_geometry = pd.read_csv(RUN_OUTPUT / 'validation_by_geometry.csv', dtype={'geometry_id': str})
geometry_summary = by_geometry.groupby(['model', 'target'], sort=False).agg(
    geometry_count=('geometry_id', 'nunique'), mean_geometry_mae=('mae', 'mean'),
    mean_geometry_rmse=('rmse', 'mean'), median_geometry_rmse=('rmse', 'median'),
    worst_geometry_rmse=('rmse', 'max'),
).reset_index()
summary = {
    'evaluation_scope': 'validation', 'selected_models': report['selected_models'],
    'selection_metric': 'validation_rmse', 'validation_rows': len(split.validation),
    'validation_geometry_count': int(split.validation.geometry_id.nunique()),
    'test_used_for_selection': False, 'per_geometry_summary': geometry_summary.to_dict('records'),
}
if not (RUN_OUTPUT / 'validation_summary.json').exists():
    temporary = RUN_OUTPUT / 'validation_summary.json.tmp'
    temporary.write_text(json.dumps(summary, indent=2, allow_nan=False) + '\n', encoding='utf-8')
    temporary.replace(RUN_OUTPUT / 'validation_summary.json')
display(geometry_summary.set_index(['target', 'model']))
print('Validation-selected variants:', report['selected_models'])

## Generalization diagnostics

Selection still uses full-row validation RMSE for consistency with notebook 03. Equal-airfoil averages and worst-airfoil errors reveal whether dense/easy polars are hiding failures elsewhere. Loss curves show the **training objective on scaled targets**, including regularization, not validation accuracy. Predictions below use at most 5,000 sampled validation rows; metric tables use all validation rows.

In [ ]:
import matplotlib.pyplot as plt

def save_plot(figure, filename):
    path = RUN_OUTPUT / filename
    if not path.exists():
        temporary = path.with_suffix('.png.tmp')
        figure.savefig(temporary, format='png', dpi=150, bbox_inches='tight')
        temporary.replace(path)

fig, axes = plt.subplots(1, len(target_columns), figsize=(6 * len(target_columns), 4), squeeze=False)
for index, target in enumerate(target_columns):
    scores = validation_results[validation_results.target == target].sort_values(['rmse', 'model'])
    axis = axes[0, index]
    axis.barh(scores.model, scores.rmse)
    axis.invert_yaxis()
    axis.set_title(f'{target}: unseen-airfoil validation RMSE')
    axis.set_xlabel('RMSE (original coefficient units)')
fig.tight_layout()
save_plot(fig, 'validation_rmse.png')
plt.show()
plt.close(fig)

fig, axis = plt.subplots(figsize=(8, 4))
for name, result in report['results'].items():
    if result['status'] == 'success':
        losses = result['training']['loss_curve']
        axis.plot(range(1, len(losses) + 1), losses, label=f'{name}: {len(losses)} iterations')
axis.set(xlabel='Training iteration', ylabel='Training objective (scaled targets)', title='Training convergence, not held-out accuracy')
axis.legend()
axis.grid(alpha=0.3)
fig.tight_layout()
save_plot(fig, 'training_loss.png')
plt.show()
plt.close(fig)

fig, axes = plt.subplots(1, len(target_columns), figsize=(6 * len(target_columns), 5), squeeze=False)
for index, target in enumerate(target_columns):
    selected = report['selected_models'][target]
    worst = by_geometry[(by_geometry.model == selected) & (by_geometry.target == target)].nlargest(10, 'rmse')
    axis = axes[0, index]
    axis.barh(worst.geometry_id.astype(str), worst.rmse)
    axis.invert_yaxis()
    axis.set_title(f'{target}: worst validation airfoils ({selected})')
    axis.set_xlabel('Per-airfoil RMSE')
fig.tight_layout()
save_plot(fig, 'worst_airfoils.png')
plt.show()
plt.close(fig)

plot_rows = split.validation.sample(n=min(5000, len(split.validation)), random_state=42)
fig, axes = plt.subplots(1, len(target_columns), figsize=(6 * len(target_columns), 5), squeeze=False)
for name in dict.fromkeys(report['selected_models'].values()):
    model = joblib.load(RUN_OUTPUT / model_artifact_path(report, name))
    predicted = np.asarray(model.predict(plot_rows[feature_columns])).reshape(len(plot_rows), len(target_columns))
    for index, target in enumerate(target_columns):
        if report['selected_models'][target] != name:
            continue
        actual = plot_rows[target].to_numpy()
        axis = axes[0, index]
        axis.scatter(actual, predicted[:, index], alpha=0.3, s=10)
        lower, upper = min(actual.min(), predicted[:, index].min()), max(actual.max(), predicted[:, index].max())
        axis.plot([lower, upper], [lower, upper], '--', color='black')
        axis.set(title=f'{target}: {name} (validation)', xlabel=f'Actual {target}', ylabel=f'Predicted {target}')
        axis.grid(alpha=0.3)
    del model, predicted
    gc.collect()
fig.tight_layout()
save_plot(fig, 'validation_predictions.png')
plt.show()
plt.close(fig)
print(f'Validation reports ready for local sync: {RUN_OUTPUT}')

## Optional final test, after decisions are frozen

Leave `RUN_FINAL_TEST = False` while choosing settings. When the experiment is finalized, set it to true and execute **only the next cell** in this live session. After reconnecting, set `EXISTING_RUN` to the saved directory and run the notebook to restore data/reports without fitting any model again. Named validation winners are loaded from the saved manifest; test metrics never select winners.

A later final-test report is synced into a separate local `<run>_with_test` snapshot, preserving the earlier validation-only snapshot. This test evaluates held-out NASA geometries, not arbitrary external airfoils without reference labels.

In [ ]:
if RUN_FINAL_TEST:
    final_results = evaluate_selected_models(
        RUN_OUTPUT, split.test[feature_columns], split.test[target_columns], target_names=target_columns,
    )
    display(final_results.set_index(['target', 'model']))
else:
    print('Test evaluation skipped. Validation-only reports are complete and can be synced locally.')